# Notebook 3: Merge agroclimático y filtrado

**Objetivo**: combinar el dataset agrícola de MIDAGRI (notebook 1) con el dataset climático de NASA POWER (notebook 2), aplicando el mapping `(región, cultivo) → distrito` para asociar cada cultivo a su clima ecológicamente coherente.

**Outputs** (todos en formato largo):
1. `dataset_por_cultivo.csv` — Análisis B: una fila por (región, cultivo, año, mes), con producción específica y clima del distrito asignado
2. `dataset_regional.csv` — Análisis A: una fila por (región, piso, año, mes), con producción agregada del piso y su clima
3. `dataset_por_cultivo_filtrado.csv` — versión del Análisis B con filtrado Pareto-80% (subset de cultivos relevantes por región)

**Decisiones metodológicas documentadas**:
- *Camino D (clima por piso ecológico)*: cada cultivo se asocia al clima del distrito de su piso ecológico, no a un promedio regional.
- *Cultivos sin asignación ecológica*: se descartan (típicamente cultivos marginales en una región).
- *Filtrado Pareto-80*: para el análisis B filtrado, se conservan los cultivos que acumulan el 80% de producción regional.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

pd.set_option('display.max_columns', 25)

## 1. Configuración

In [7]:
# >>>>>>>>>>> AJUSTAR ESTAS RUTAS <<<<<<<<<<<
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "SCRIPTS":
    ROOT = ROOT.parent

RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_BASE = ROOT / "Datasets Originales"
RUTA_MAPPING = ROOT / "BDS" / "mapping_cultivo_distrito.csv"
RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)

RUTA_MIDAGRI = RUTA_OUTPUT / "midagri_largo.csv"
RUTA_NASA    = RUTA_OUTPUT / "nasa_2020_2025.csv"

# Inputs (deben existir tras correr notebooks 1 y 2)
RUTA_MIDAGRI = RUTA_OUTPUT / "midagri_largo.csv"
RUTA_NASA    = RUTA_OUTPUT / "nasa_2020_2025.csv"
RUTA_MAPPING = RUTA_OUTPUT / "mapping_cultivo_distrito.csv"

# Verificar que existen
for ruta in [RUTA_MIDAGRI, RUTA_NASA, RUTA_MAPPING]:
    assert ruta.exists(), f"No existe: {ruta}"

print("Inputs OK")

Inputs OK


## 2. Cargar los tres insumos

In [8]:
df_midagri = pd.read_csv(RUTA_MIDAGRI)
df_nasa    = pd.read_csv(RUTA_NASA)
df_mapping = pd.read_csv(RUTA_MAPPING)

print(f"MIDAGRI:  {df_midagri.shape}")
print(f"NASA:     {df_nasa.shape}")
print(f"Mapping:  {df_mapping.shape}")

# Limpiar columnas vacías del mapping (las de revisión que estaban vacías)
df_mapping = df_mapping[['region','cultivo','piso_asignado','distrito']].copy()

# Verificar consistencia de regiones
print(f"\nRegiones MIDAGRI: {sorted(df_midagri['region'].unique())}")
print(f"Regiones NASA:    {sorted(df_nasa['region'].unique())}")
print(f"Regiones Mapping: {sorted(df_mapping['region'].unique())}")

MIDAGRI:  (23328, 6)
NASA:     (1008, 19)
Mapping:  (200, 7)

Regiones MIDAGRI: ['Ica', 'Junin', 'La Libertad', 'Piura', 'Puno', 'San Martin']
Regiones NASA:    ['Ica', 'Junin', 'La Libertad', 'Piura', 'Puno', 'San Martin']
Regiones Mapping: ['Ica', 'Junin', 'La Libertad', 'Piura', 'Puno', 'San Martin']


In [ ]:
# Compatibilidad tras exportación con nombres legibles (solo insumos ya mergeados)
def _restaurar_nasa(df):
    _R = {
        "temp_promedio": "t2m", "temp_maxima": "t2m_max", "temp_minima": "t2m_min",
        "precipitacion": "prectotcorr", "humedad_relativa": "rh2m",
        "radiacion_solar": "allsky_sfc_sw_dwn", "velocidad_viento": "ws2m",
        "presion_atmosferica": "ps", "humedad_suelo": "gwetroot",
        "temp_superficie": "ts", "punto_rocio": "t2mdew", "humedad_especifica": "qv2m",
    }
    cols = {k: v for k, v in _R.items() if k in df.columns}
    return df.rename(columns=cols) if cols else df


## 3. Análisis B — Dataset por cultivo

Tres pasos:
1. Hacer left join `MIDAGRI ⨝ Mapping` para asignar distrito a cada (región, cultivo).
2. Hacer left join con NASA usando `(distrito, año, mes)`.
3. Limpiar y validar.

In [9]:
# Paso 1: Asignar distrito climático a cada (region, cultivo)
df_b = df_midagri.merge(
    df_mapping,
    on=['region', 'cultivo'],
    how='left'
)

# Cuántos cultivos tienen asignación
n_sin_asig = df_b['distrito'].isna().sum()
print(f"Filas totales: {len(df_b)}")
print(f"Filas con distrito asignado: {(~df_b['distrito'].isna()).sum()}")
print(f"Filas SIN distrito asignado (cultivos no compatibles ecológicamente): {n_sin_asig}")

# Las filas sin distrito son cultivos que no tienen piso compatible en su región
# (ej. cacao en Ica). Su producción típicamente es 0 o NaN. Las descartamos.
df_b = df_b.dropna(subset=['distrito']).copy()
print(f"\nFilas tras descartar sin asignación: {len(df_b)}")

Filas totales: 23328
Filas con distrito asignado: 14400
Filas SIN distrito asignado (cultivos no compatibles ecológicamente): 8928

Filas tras descartar sin asignación: 14400


In [10]:
# Paso 2: Hacer merge con NASA
# Renombrar columnas de NASA para evitar conflictos en el merge
df_nasa_para_merge = df_nasa.rename(columns={'region':'region_nasa'})

df_b = df_b.merge(
    df_nasa_para_merge,
    on=['distrito','anio','mes_num'],
    how='left'
)

# Validar: cada fila debe tener clima
n_sin_clima = df_b[df_b['t2m'].isna()].shape[0]
print(f"Filas sin clima asociado: {n_sin_clima}")
if n_sin_clima > 0:
    print("(Esto pasa solo si hay desajuste año/mes entre MIDAGRI y NASA)")

# Limpiar columnas redundantes
cols_redundantes = ['region_nasa','lat','lon','piso']  # 'piso' viene de NASA pero ya tenemos 'piso_asignado' de MIDAGRI  # tenemos region desde MIDAGRI, lat/lon no son necesarios para análisis
df_b = df_b.drop(columns=[c for c in cols_redundantes if c in df_b.columns])

# Reordenar columnas
cols_meta = ['region','piso_asignado','distrito','cultivo','anio','mes_num','mes_nombre','produccion_mensual']
cols_clima = [c for c in df_b.columns if c not in cols_meta]
df_b = df_b[cols_meta + cols_clima]

print(f"\nDataset B final: {df_b.shape}")
print(f"Columnas: {list(df_b.columns)}")
print(f"\nMuestra:")
print(df_b.head(10).to_string(index=False))

Filas sin clima asociado: 0

Dataset B final: (14400, 20)
Columnas: ['region', 'piso_asignado', 'distrito', 'cultivo', 'anio', 'mes_num', 'mes_nombre', 'produccion_mensual', 't2m', 't2m_max', 't2m_min', 'prectotcorr', 'rh2m', 'allsky_sfc_sw_dwn', 'ws2m', 'ps', 'gwetroot', 'ts', 't2mdew', 'qv2m']

Muestra:
region piso_asignado     distrito          cultivo  anio  mes_num mes_nombre  produccion_mensual   t2m  t2m_max  t2m_min  prectotcorr  rh2m  allsky_sfc_sw_dwn  ws2m    ps  gwetroot    ts  t2mdew  qv2m
   Ica         costa Chincha Alta         aceituna  2020        1      Enero                0.00 21.96    27.55    18.21         0.12 78.45              22.25  2.99 95.82      0.33 23.95   17.83 13.41
   Ica         costa Chincha Alta              aji  2020        1      Enero              635.00 21.96    27.55    18.21         0.12 78.45              22.25  2.99 95.82      0.33 23.95   17.83 13.41
   Ica         costa Chincha Alta              ajo  2020        1      Enero              

## 4. Análisis A — Dataset por piso ecológico

Para cada combinación `(region, piso, anio, mes)`:
1. **Producción**: suma de todos los cultivos asignados a ese piso.
2. **Clima**: el del distrito de ese piso (todos los cultivos del mismo piso comparten distrito → un solo clima por fila).

In [17]:
# Agregar producción por piso
df_a_prod = (df_b.groupby(['region','piso_asignado','distrito','anio','mes_num','mes_nombre'])
             .agg(produccion_total_piso=('produccion_mensual','sum'),
                  n_cultivos=('cultivo','nunique'))
             .reset_index())

# Agregar el clima (un solo registro climático por (distrito, anio, mes))
cols_clima = [c for c in df_b.columns if c not in 
              ['region','piso_asignado','distrito','cultivo','anio','mes_num','mes_nombre','produccion_mensual']]

df_clima_distrito = df_b[['distrito','anio','mes_num'] + cols_clima].drop_duplicates(['distrito','anio','mes_num'])

df_a = df_a_prod.merge(df_clima_distrito, on=['distrito','anio','mes_num'], how='left')

# Renombrar columna de piso
df_a = df_a.rename(columns={'piso_asignado':'piso'})

# Reordenar
cols_meta_a = ['region','piso','distrito','anio','mes_num','mes_nombre','n_cultivos','produccion_total_piso']
df_a = df_a[cols_meta_a + cols_clima]

print(f"Dataset A (regional por piso): {df_a.shape}")
print(f"Unidades únicas (region, piso): {df_a[['region','piso']].drop_duplicates().shape[0]}")
print(f"\nUnidades:")
unidades = df_a[['region','piso','distrito']].drop_duplicates().sort_values(['region','piso'])
print(unidades.to_string(index=False))

Dataset A (regional por piso): (1008, 20)
Unidades únicas (region, piso): 14

Unidades:
     region               piso     distrito
        Ica              costa Chincha Alta
      Junin         selva_alta       Perene
      Junin         selva_baja    Rio Tambo
      Junin             sierra     El Tambo
La Libertad              costa         Viru
La Libertad             sierra   Huamachuco
La Libertad              yunga       Cascas
      Piura        bosque_seco  Tambogrande
      Piura             sierra    Canchaque
      Piura        valle_chira      Sullana
       Puno altiplano_lacustre        Ilave
       Puno          puna_alta      Ayaviri
 San Martin    selva_alto_mayo    Moyobamba
 San Martin     selva_huallaga      Tocache


## 5. Filtrado Pareto-80%

Para el dataset por cultivo (B), generamos una versión filtrada que conserva solo los cultivos que acumulan el **80% de la producción** en cada región. Esto:
- Reduce ruido (cultivos marginales con producción ínfima)
- Enfoca el análisis en los cultivos económicamente relevantes
- Es un criterio reproducible y documentable

In [18]:
def filtrar_pareto(df_b, umbral=0.80):
    """Para cada región, conserva los cultivos que acumulan hasta `umbral` de la producción total."""
    cultivos_a_conservar = []
    
    for region in sorted(df_b['region'].unique()):
        # Producción total acumulada por cultivo (todos los meses, todos los años)
        prod_por_cultivo = (df_b[df_b['region']==region]
                            .groupby('cultivo')['produccion_mensual']
                            .sum().sort_values(ascending=False))
        prod_por_cultivo = prod_por_cultivo[prod_por_cultivo > 0]
        
        if len(prod_por_cultivo) == 0:
            continue
        
        # Acumulado proporcional
        acum = prod_por_cultivo.cumsum() / prod_por_cultivo.sum()
        # Cultivos hasta llegar al umbral (incluyendo el que cruza)
        cultivos_top = acum[acum <= umbral].index.tolist()
        # Asegurar al menos uno
        if len(cultivos_top) == 0 and len(acum) > 0:
            cultivos_top = [acum.index[0]]
        # Si el primer cultivo ya supera el umbral, igual lo agregamos
        if acum.iloc[0] > umbral:
            cultivos_top = [acum.index[0]]
        
        for c in cultivos_top:
            cultivos_a_conservar.append((region, c))
        
        print(f"{region:<13}: {len(cultivos_top)} cultivos ({100*acum.loc[cultivos_top[-1]]:.1f}% de producción acumulada)")
    
    # Filtrar
    cultivos_set = set(cultivos_a_conservar)
    df_filtrado = df_b[df_b.apply(lambda r: (r['region'], r['cultivo']) in cultivos_set, axis=1)].copy()
    return df_filtrado, cultivos_a_conservar

print("=== FILTRADO PARETO-80 (por región) ===\n")
df_b_filtrado, cultivos_pareto = filtrar_pareto(df_b, umbral=0.80)
print(f"\nDataset B filtrado: {df_b_filtrado.shape}")
print(f"Total combinaciones (region, cultivo) conservadas: {len(cultivos_pareto)}")

=== FILTRADO PARETO-80 (por región) ===

Ica          : 6 cultivos (74.4% de producción acumulada)
Junin        : 7 cultivos (77.5% de producción acumulada)
La Libertad  : 3 cultivos (78.8% de producción acumulada)
Piura        : 3 cultivos (79.8% de producción acumulada)
Puno         : 2 cultivos (76.4% de producción acumulada)
San Martin   : 3 cultivos (76.2% de producción acumulada)

Dataset B filtrado: (1728, 20)
Total combinaciones (region, cultivo) conservadas: 24


## 6. Validaciones finales

Tres validaciones críticas:
1. Coherencia entre A y B: la suma de B por piso debe coincidir con la producción de A.
2. Cobertura temporal completa.
3. Sin NaN en variables climáticas (los meses faltantes de MIDAGRI quedaron como NaN en producción, pero el clima debe estar siempre).

In [19]:
# Validación 1: coherencia A vs B
print("=== Validación 1: coherencia A vs B ===")
suma_b = df_b.groupby(['region','piso_asignado','anio','mes_num'])['produccion_mensual'].sum().reset_index()
suma_b = suma_b.rename(columns={'piso_asignado':'piso','produccion_mensual':'suma_desde_B'})
check = df_a[['region','piso','anio','mes_num','produccion_total_piso']].merge(
    suma_b, on=['region','piso','anio','mes_num']
)
check['diferencia'] = (check['produccion_total_piso'] - check['suma_desde_B']).abs()
max_diff = check['diferencia'].max()
print(f"Máxima diferencia entre A y suma de B: {max_diff:.6f} (debe ser 0)")
assert max_diff < 1e-6, "Inconsistencia entre datasets A y B"
print("✓ Datasets A y B son coherentes\n")

# Validación 2: cobertura temporal
print("=== Validación 2: cobertura temporal ===")
print(f"Años en B: {sorted(df_b['anio'].unique())}")
print(f"Meses en B: {sorted(df_b['mes_num'].unique())}")

# Validación 3: NaN climaticos
print("\n=== Validación 3: NaN en variables climáticas ===")
cols_clima_check = [c for c in df_b.columns if c in ['t2m','t2m_max','t2m_min','prectotcorr','rh2m','allsky_sfc_sw_dwn','ws2m','ps','gwetroot','ts','t2mdew','qv2m']]
nan_clima = df_b[cols_clima_check].isna().sum().sum()
print(f"Total NaN en variables climáticas: {nan_clima} (esperado: 0)")
if nan_clima > 0:
    print("Detalle por variable:")
    for c in cols_clima_check:
        n = df_b[c].isna().sum()
        if n > 0:
            print(f"  {c}: {n}")

=== Validación 1: coherencia A vs B ===
Máxima diferencia entre A y suma de B: 0.000000 (debe ser 0)
✓ Datasets A y B son coherentes

=== Validación 2: cobertura temporal ===
Años en B: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Meses en B: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]

=== Validación 3: NaN en variables climáticas ===
Total NaN en variables climáticas: 0 (esperado: 0)


## 7. Exportar

In [20]:
ruta_b = RUTA_OUTPUT / "dataset_por_cultivo.csv"
ruta_b_filt = RUTA_OUTPUT / "dataset_por_cultivo_filtrado.csv"
ruta_a = RUTA_OUTPUT / "dataset_regional.csv"

df_b.to_csv(ruta_b, index=False, encoding='utf-8-sig')
df_b_filtrado.to_csv(ruta_b_filt, index=False, encoding='utf-8-sig')
df_a.to_csv(ruta_a, index=False, encoding='utf-8-sig')

print(f"Exportados:")
print(f"  - {ruta_a}  ({len(df_a):,} filas)  [Análisis A: clima x piso]")
print(f"  - {ruta_b}  ({len(df_b):,} filas)  [Análisis B: clima x cultivo]")
print(f"  - {ruta_b_filt}  ({len(df_b_filtrado):,} filas)  [Análisis B filtrado Pareto-80]")

print(f"\n=== LISTO ===")
print(f"Próximo paso: notebooks 4 y 5 de EDA (uno para cada análisis)")

Exportados:
  - C:\Users\Joyssie\Documents\Data Minning\Proyecto\Outputs\dataset_regional.csv  (1,008 filas)  [Análisis A: clima x piso]
  - C:\Users\Joyssie\Documents\Data Minning\Proyecto\Outputs\dataset_por_cultivo.csv  (14,400 filas)  [Análisis B: clima x cultivo]
  - C:\Users\Joyssie\Documents\Data Minning\Proyecto\Outputs\dataset_por_cultivo_filtrado.csv  (1,728 filas)  [Análisis B filtrado Pareto-80]

=== LISTO ===
Próximo paso: notebooks 4 y 5 de EDA (uno para cada análisis)
